In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

In [2]:
class MEG_CNN_LSTM(nn.Module):
    def __init__(
        self,
        n_sensors=248,       # 感測器數量
        n_classes=4,         # rest, math, memory, motor
        downsample_factor=10,  # 每幾步取1，可調參數
        cnn_filters=[64, 128],
        kernel_size=3,
        lstm_hidden=128,
        lstm_layers=2,
        dropout=0.5,
    ):
        super().__init__()

        self.downsample_factor = downsample_factor

        # CNN 部分：在時間軸上抽局部特徵
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv1d(in_channels=n_sensors, out_channels=cnn_filters[0], kernel_size=kernel_size, padding=1),
            nn.BatchNorm1d(cnn_filters[0]),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),  # 時間步減半
            nn.Dropout(dropout),

            # Block 2
            nn.Conv1d(in_channels=cnn_filters[0], out_channels=cnn_filters[1], kernel_size=kernel_size, padding=1),
            nn.BatchNorm1d(cnn_filters[1]),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),  # 時間步再減半
            nn.Dropout(dropout),
        )

        # LSTM 部分：學時序依賴
        self.lstm = nn.LSTM(
            input_size=cnn_filters[1],   # CNN輸出的特徵數
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0,
        )

        # 分類頭
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        # x shape: (batch, n_sensors, time_steps) = (batch, 248, 35624)

        # 1. Downsample：每 downsample_factor 步取1
        x = x[:, :, ::self.downsample_factor]
        # shape: (batch, 248, 35624/downsample_factor)

        # 2. CNN：input 要是 (batch, channels, length)
        #    這裡 channels = n_sensors = 248，length = 時間步
        x = self.cnn(x)
        # shape: (batch, 128, 時間步/4)

        # 3. LSTM：input 要是 (batch, time_steps, features)
        x = x.permute(0, 2, 1)
        # shape: (batch, 時間步/4, 128)

        x, _ = self.lstm(x)
        # shape: (batch, 時間步/4, lstm_hidden)

        # 4. 取最後一個時間步的輸出來分類
        x = x[:, -1, :]
        # shape: (batch, lstm_hidden)

        # 5. 分類
        out = self.classifier(x)
        # shape: (batch, 4)

        return out

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    total_correct = 0
    total_samples = 0
 
    for x, y in loader:
        x, y = x.to(device), y.to(device)
 
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
 
        total_loss += loss.item() * len(y)
        total_correct += (out.argmax(dim=1) == y).sum().item()
        total_samples += len(y)
 
    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy
 
 
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
 
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
 
            total_loss += loss.item() * len(y)
            total_correct += (out.argmax(dim=1) == y).sum().item()
            total_samples += len(y)
 
    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy
 
 
def train(
    model,
    train_loader,
    val_loader,
    n_epochs=30,
    lr=1e-3,
    device=None,
    save_path="best_model.pt",
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
 
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
 
    # 學習率 scheduler：val loss 沒進步就降低 lr
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=3, factor=0.5, verbose=True
    )
 
    best_val_loss = float("inf")
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
 
    for epoch in range(1, n_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
 
        scheduler.step(val_loss)
 
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
 
        print(
            f"Epoch {epoch:3d}/{n_epochs} | "
            f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}"
        )
 
        # 儲存最好的 model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"  → Best model saved!")
 
    print(f"\nTraining done. Best val loss: {best_val_loss:.4f}")
    return history


In [1]:
# 步驟 1: 導入數據預處理模組
from data_preprocessing import load_datasets
import numpy as np

# 步驟 2: 設定數據路徑並加載數據
base_dir = r"C:\Users\2035k\IAN\Master\Deep_Learning\Final Project data"
downsample_factor = 10

# 這會自動加載、下採樣、正規化所有數據
datasets = load_datasets(base_dir, downsample_factor=downsample_factor, verbose=True)


Processing folder: C:\Users\2035k\IAN\Master\Deep_Learning\Final Project data\Intra\train
Number of h5 files: 32
X shape: (32, 248, 3562)
y shape: (32,)
Label distribution: Counter({np.int64(0): 8, np.int64(3): 8, np.int64(1): 8, np.int64(2): 8})

Processing folder: C:\Users\2035k\IAN\Master\Deep_Learning\Final Project data\Intra\test
Number of h5 files: 8
X shape: (8, 248, 3562)
y shape: (8,)
Label distribution: Counter({np.int64(0): 2, np.int64(3): 2, np.int64(1): 2, np.int64(2): 2})

Processing folder: C:\Users\2035k\IAN\Master\Deep_Learning\Final Project data\Cross\train
Number of h5 files: 64
X shape: (64, 248, 3562)
y shape: (64,)
Label distribution: Counter({np.int64(0): 16, np.int64(3): 16, np.int64(1): 16, np.int64(2): 16})

Processing folder: C:\Users\2035k\IAN\Master\Deep_Learning\Final Project data\Cross\test1
Number of h5 files: 16
X shape: (16, 248, 3562)
y shape: (16,)
Label distribution: Counter({np.int64(0): 4, np.int64(3): 4, np.int64(1): 4, np.int64(2): 4})

Process

In [ ]:
if __name__ == "__main__":
    from torch.utils.data import TensorDataset
 
    # ========================================
    # 方式 1: 使用 Cross-subject 訓練/測試
    # ========================================
    
    # 加載數據（已經自動下採樣和正規化）
    X_train = datasets["cross_train"]["X"]  # shape: (N, 248, time_steps)
    y_train = datasets["cross_train"]["y"]  # shape: (N,)
    
    X_test = datasets["cross_test1"]["X"]
    y_test = datasets["cross_test1"]["y"]
    
    print(f"Train data shape: {X_train.shape}")
    print(f"Train labels shape: {y_train.shape}")
    print(f"Test data shape: {X_test.shape}")
    print(f"Test labels shape: {y_test.shape}\n")
    
    # 轉換成 PyTorch tensor
    x_train_tensor = torch.from_numpy(X_train).float()
    y_train_tensor = torch.from_numpy(y_train).long()
    
    x_test_tensor = torch.from_numpy(X_test).float()
    y_test_tensor = torch.from_numpy(y_test).long()
    
    # 建立 dataset
    train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(x_test_tensor, y_test_tensor)
    
    # 如果要進一步分割訓練集為 train/val
    train_size = int(0.8 * len(train_dataset))
    val_size = len(train_dataset) - train_size
    train_set, val_set = torch.utils.data.random_split(train_dataset, [train_size, val_size])
    
    # 建立 DataLoader
    train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=8)
    test_loader = DataLoader(test_dataset, batch_size=8)
    
    # 建立模型
    model = MEG_CNN_LSTM(
        n_sensors=248,
        n_classes=4,
        downsample_factor=1,  # 已經在 data_preprocessing 中下採樣過，所以這裡設 1
        cnn_filters=[64, 128],
        lstm_hidden=128,
        lstm_layers=2,
        dropout=0.5
    )
    
    # 訓練
    print("=" * 60)
    print("開始訓練...")
    print("=" * 60)
    
    history = train(
        model,
        train_loader,
        val_loader,
        n_epochs=30,
        lr=1e-3,
        save_path="best_meg_model.pt",
    )
    
    # 評估測試集
    print("\n" + "=" * 60)
    print("評估測試集...")
    print("=" * 60)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")


downsample_factor= 5 | output shape: torch.Size([8, 4]) | params: 345,476
downsample_factor=10 | output shape: torch.Size([8, 4]) | params: 345,476
downsample_factor=20 | output shape: torch.Size([8, 4]) | params: 345,476
